# Hardware-Aware Model Architecture & Execution Mapper

End-to-end demo: profile a model on an NVIDIA GPU and visualize across all 3 tabs.

- **Tab 1** — Workload Dashboard (per-request TTFT, decode, queue, scatter plots, sortable table)
- **Tab 2** — Profiler (nn.Module → op → kernel hierarchy for Fast/Typical/Slow representatives)
- **Tab 3** — Architecture (one-time model structure with layer collapse)

All commands run on the **GPU host** unless noted otherwise.

## Prerequisites

You can skip this step if you already have vllm installed. See: https://docs.vllm.ai/en/latest/getting_started/quickstart/#nvidia-cuda

### 1. Clone vllm repo
```bash
if [ ! -d "vllm" ]; then
  git clone https://github.com/vllm-project/vllm.git
fi
```

### 2. Create virtual environment
```bash
uv venv --python 3.12 --seed
source .venv/bin/activate
```

### 3. Install packages
```bash
uv pip install torch --torch-backend=auto 
uv pip install setuptools wheel setuptools-rust
uv pip install setuptools_scm ninja cmake numpy
cd vllm
MAX_JOBS=12 pip install -vv --no-build-isolation -e .
cd ../
```

### 4. Verify
```bash
python -c "import vllm; print(vllm.__version__)"
nvidia-smi
```

---
## Tab 1: Workload Dashboard

Produces `workload_data.json`.

Run `vllm serve` with a benchmark client to capture per-request TTFT, decode, queue, and ITL timing across many requests. Prometheus provides server-side aggregate metrics. `parse_workload.py` merges everything into one JSON.

The dashboard shows scatter plots (TTFT vs input length, Decode vs output length), a request timeline, summary stats, and a color-coded sortable table with CSV download.

In [ ]:
# Set model and working directory
import os
os.chdir("/home/redhat-et/lkesem/vllm_arch_map")
print("CURRENT DIRECTORY:", os.getcwd())

os.environ["MODEL"] = "Qwen/Qwen3-8B"

# Start vLLM server in background
import subprocess, time, urllib.request
os.makedirs("benchmark_outputs", exist_ok=True)

server = subprocess.Popen(
    ["vllm", "serve", os.environ["MODEL"],
     "--max-model-len", "8192",
     "--gpu-memory-utilization", "0.85"],
    stdout=open("server_log.txt", "w"), stderr=subprocess.STDOUT
)

for i in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health")
        print(f"Server ready! (pid={server.pid})")
        break
    except:
        time.sleep(2)
        if i % 10 == 9:
            print(f"Still waiting... ({(i+1)*2}s)")

In [ ]:
# Run benchmark in background + capture Prometheus mid-run for KV cache usage
import os, subprocess, time
os.environ["MODEL"] = "Qwen/Qwen3-8B"

bench = subprocess.Popen(
    "vllm bench serve --backend vllm --model $MODEL "
    "--base-url http://localhost:8000 "
    "--endpoint /v1/completions --dataset-name random "
    "--num-prompts 100 --max-concurrency 3 "
    "--random-input-len 512 --random-output-len 128 "
    "--random-range-ratio 0.5 "
    "--save-result --save-detailed --result-dir benchmark_outputs/",
    shell=True, env=os.environ
)

# Wait for requests to be in-flight, then capture Prometheus (KV cache will be non-zero)
time.sleep(60)
subprocess.run("curl -s http://localhost:8000/metrics > benchmark_outputs/prometheus.txt",shell=True, check=True)
print("Prometheus metrics captured mid-run.")

bench.wait()
print(f"Benchmark finished (exit={bench.returncode}).")

In [ ]:
# Stop server + cleanup
server.terminate()
server.wait()
print("Server stopped.")
!pkill -f "vllm serve" 2>/dev/null || true
print("Cleaned up all vllm processes.")

In [ ]:
# Merge benchmark JSON + Prometheus → workload_data.json
import glob
bench_files = sorted(glob.glob("benchmark_outputs/*.json"))
if bench_files:
    bench_file = bench_files[-1]  # latest benchmark
    print(f"Using: {bench_file}")
else:
    raise FileNotFoundError("No benchmark JSON found in benchmark_outputs/")

!python vllm_arch_map/parse_workload.py \
    --bench "{bench_file}" \
    --prometheus benchmark_outputs/prometheus.txt \
    --dataset-name random \
    -o samples/workload_data.json

---
## Tab 2: Profiler (Per-Request Module Breakdown)

Produces `per_request_profiles.json`.

`profile_serve.py --workload-json` reads `workload_data.json` from Tab 1 and auto-picks 3 representative requests:
- **Fast** (p10 by latency) — short input, quick response
- **Typical** (p50) — median behavior  
- **Slow** (p90) — long input, tail latency

Each representative is profiled offline with `vllm.LLM` + `torch.profiler`, producing a Chrome trace with the full `nn.Module` → CPU op → GPU kernel hierarchy. The Profiler tab shows the module tree with timing columns for each representative — you can immediately see which modules scale badly with input length.

**Why compress to JSON?** Raw Chrome traces are 50-100 MB each. The processed `per_request_profiles.json` is ~1 MB — only the module tree structure, timing, and kernel mappings that the viewer needs.

In [ ]:
# Profile: auto-pick Fast/Typical/Slow from workload data
import os
os.environ["MODEL"] = "Qwen/Qwen3-8B"

!python vllm_arch_map/profile_serve.py $MODEL \
    --workload-json samples/workload_data.json

In [ ]:
# Verify traces + process into merged module tree
!ls -lh traces/per_request/request_*.json*
!echo "---"
!python vllm_arch_map/profiler_to_diagram.py \
    --per-request-dir traces/per_request --out_dir samples/

---
## Tab 3: Architecture (one-time per model)

Single terminal. Produces `model_execution.json`.

`walk_model.py` walks the `nn.Module` tree and captures a PyTorch profiler trace. The trace is saved as a compressed `.json.gz` file (raw Chrome trace format). `profiler_to_diagram.py` reads it directly (no manual decompression needed) and builds the module hierarchy with kernel mappings.

In [ ]:
# Walk model tree + capture profiler trace
import os
os.environ["MODEL"] = "Qwen/Qwen3-8B"

!VLLM_ALLOW_INSECURE_SERIALIZATION=1 python vllm_arch_map/walk_model.py $MODEL \
    -o samples/model_tree.json --with-forward --profile-dir ./samples/traces/

In [ ]:
# Process: build module tree with kernel correlation + layer collapse
!python vllm_arch_map/profiler_to_diagram.py \
    --out_dir samples/ --traces-dir samples/traces

---
## View

### On the GPU host
```bash
python3 -m http.server 8765 --directory samples
```

### Or SCP to local machine
```bash
scp host:~/vllm_arch_map/samples/per_request_profiles.json samples/
scp host:~/vllm_arch_map/samples/workload_data.json samples/
scp host:~/vllm_arch_map/samples/model_execution.json samples/

python3 -m http.server 8765 --directory samples
open http://localhost:8765/model_viz.html
```

---
## Output Files

| File | Tab | Size | Contents |
|---|---|---|---|
| `workload_data.json` | Tab 1: Workload | ~50 KB | Per-request TTFT/decode/queue, Prometheus metrics |
| `per_request_profiles.json` | Tab 2: Profiler | ~1 MB | Merged module tree with Fast/Typical/Slow timing, kernel mapping, layer variance |
| `model_execution.json` | Tab 3: Architecture | ~300 KB | Module hierarchy, op→kernel correlation, layer collapse |